In [0]:
from pyspark.sql import functions as F

# 1. Leitura das tabelas Silver
df_est = spark.table("workspace.default.silver_estabelecimentos")
df_emp = spark.table("workspace.default.silver_empresas")
df_soc = spark.table("workspace.default.silver_socios")
df_smp = spark.table("workspace.default.silver_simples_nacional")
df_mun = spark.table("workspace.default.silver_dominio_municipios")
df_nat = spark.table("workspace.default.silver_dominio_naturezas_juridicas")
df_cna = spark.table("workspace.default.silver_dominio_cnaes")

# 2. Agregação de sócios por empresa (CNPJ Básico)
df_socios_agg = (
    df_soc
    .groupBy("nr_cnpj_basico")
    .agg(F.count("id_socio_hash").alias("qt_socios_registrados"))
)

# 3. Cruzamento e Tradução Semântica
df_gold_dp_estabelecimentos = (
    df_est.alias("est")
    .join(df_emp.alias("emp"), "nr_cnpj_basico", "inner")
    .join(df_smp.alias("smp"), "nr_cnpj_basico", "left")
    .join(df_socios_agg.alias("soc"), "nr_cnpj_basico", "left")
    .join(df_mun.alias("mun"), F.col("est.cd_municipio") == F.col("mun.cd_municipio"), "left")
    .join(df_nat.alias("nat"), F.col("emp.cd_natureza_juridica") == F.col("nat.cd_natureza_juridica"), "left")
    .join(df_cna.alias("cna"), F.col("est.cd_cnae_fiscal_principal") == F.col("cna.cd_cnae"), "left")
    .select(
        # Chaves mantidas como BIGINT para o teste de transição de contrato
        F.col("est.nr_cnpj_completo").alias("nr_cnpj"),
        F.col("est.nr_cnpj_basico"),
        F.col("emp.nm_razao_social"),
        F.col("est.nm_fantasia"),
        F.when(F.col("est.cd_identificador_matriz_filial") == 1, "MATRIZ")
         .when(F.col("est.cd_identificador_matriz_filial") == 2, "FILIAL")
         .otherwise("OUTROS").alias("tp_estabelecimento"),
        F.when(F.col("est.cd_situacao_cadastral") == "01", "NULA")
         .when(F.col("est.cd_situacao_cadastral") == "02", "ATIVA")
         .when(F.col("est.cd_situacao_cadastral") == "03", "SUSPENSA")
         .when(F.col("est.cd_situacao_cadastral") == "04", "INAPTA")
         .when(F.col("est.cd_situacao_cadastral") == "08", "BAIXADA")
         .otherwise("DESCONHECIDO").alias("ds_situacao_cadastral"),
        F.col("est.dt_inicio_atividade"),
        F.col("est.dt_situacao_cadastral"),
        F.col("emp.vl_capital_social"),
        F.col("nat.ds_natureza_juridica"),
        F.when(F.col("emp.cd_porte_empresa") == "01", "MICRO EMPRESA")
         .when(F.col("emp.cd_porte_empresa") == "03", "EMPRESA DE PEQUENO PORTE")
         .when(F.col("emp.cd_porte_empresa") == "05", "DEMAIS")
         .otherwise("NAO INFORMADO").alias("ds_porte_empresa"),
        F.when(F.col("smp.fl_opcao_simples") == "S", True).otherwise(False).alias("fl_simples_nacional_ativo"),
        F.when(F.col("smp.fl_opcao_mei") == "S", True).otherwise(False).alias("fl_mei_ativo"),
        F.col("est.cd_cnae_fiscal_principal"),
        F.col("cna.ds_cnae").alias("ds_cnae_fiscal_principal"),
        F.col("est.ds_cnae_fiscal_secundaria"),
        F.concat_ws(" ", F.col("est.ds_tipo_logradouro"), F.col("est.nm_logradouro"), F.col("est.nr_logradouro")).alias("ds_logradouro_completo"),
        F.col("est.nm_bairro"),
        F.col("est.nr_cep"),
        F.col("est.sg_uf"),
        F.col("mun.ds_municipio").alias("nm_municipio"),
        F.coalesce(F.col("soc.qt_socios_registrados"), F.lit(0)).alias("qt_socios_registrados"),
        F.current_timestamp().alias("dt_carga_gold")
    )
)

(
    df_gold_dp_estabelecimentos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("sg_uf")
    .saveAsTable("workspace.default.gold_dp_cnpj_estabelecimentos_360")
)
print("-> workspace.default.gold_dp_cnpj_estabelecimentos_360 gravada.")

-> workspace.default.gold_dp_cnpj_estabelecimentos_360 gravada.
